# Validation sample for the author attribution

This notebook draws the sample of attributions that gets checked by hand,
and documents that the draw is random and reproducible.

**Why this exists.** The data paper reports how many opinions the attribution
resolves. A reviewer will ask the harder question: of the ones it resolves,
how many are *right*? That can only be answered by reading opinions and
comparing them against the match. This notebook produces the file for that,
and leaves an auditable record of how the rows were chosen -- a sample picked
by hand, or by a method nobody can re-run, is not evidence.

**Nothing here writes to the database.** Every query is a `SELECT`.

**Which database.** The notebook reads a restore of the published release, not
the working database the project was built in. Restore the Zenodo deposit into
an empty database with `restore_deposit.sh` and point `RESTORE_DB` at it. The
draw and the sample are then reproducible by anyone who has the deposit.

In [ ]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import pandas as pd
import numpy as np
import psycopg2

def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'restore_deposit.sh').is_file():
            return candidate
    raise RuntimeError('Run the notebook from inside the repository')


BASE = find_repository_root()
OUT = Path(os.environ.get(
    'VALIDATION_SAMPLE_OUT',
    BASE / 'validation/author_attribution_validation_sample_to_annotate.csv',
))

# The notebook reads the published release, not the working database it
# was built in. Restore the Zenodo package into an empty database with
# restore_deposit.sh, then point RESTORE_DB at it. Everything below then
# runs against exactly the data the paper describes.
#
#   createdb -h <socket> -p <port> courtlistener_release
#   RESTORE_DB=courtlistener_release ./restore_deposit.sh
#
# Read-only throughout.
RESTORE_DB = os.environ.get('RESTORE_DB', 'courtlistener_release_restore')
RESTORE_HOST = os.environ.get('RESTORE_HOST', '')
RESTORE_PORT = int(os.environ.get('RESTORE_PORT', '5432'))
DSN = dict(dbname=RESTORE_DB, port=RESTORE_PORT)
if RESTORE_HOST:
    DSN['host'] = RESTORE_HOST

# pandas warns that it only supports SQLAlchemy connectables. A psycopg2
# connection works for plain SELECTs and keeps the dependency list short,
# so the warning is silenced deliberately rather than papered over.
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

# The salt fixes the draw. Changing it draws a different sample, which is why
# it is written down here rather than generated.
SALT = 'epjds-validation-2026'

# Rows per attribution step. Not equal, and not proportional to size -- see
# the allocation section below.
ALLOCATION = {1: 20, 2: 60, 3: 60, 4: 60}

pd.set_option('display.width', 120)

## 1. The population we sample from

Only attributions that resolve to **exactly one judge** are eligible. Where
several candidates share the recorded gender the attribution is usable for
gender analysis but there is no single person for an annotator to confirm,
so those rows cannot be checked this way.

The released attribution table carries the `person_unique` flag produced by
`deposit_pipeline/05_deposit/01_unique_person_attribution.sql`. The notebook
uses that flag directly, so the sampling frame is exactly the 524,777
opinions reported as identifying one person.

In [ ]:
POPULATION_SQL = '''
-- The released table already carries the step, tier and person-unique flag.
SELECT
    a.opinion_id,
    a.court_id,
    a.year_filed,
    a.gender,
    a.match_tier       AS tier,
    a.match_tier_label AS tier_label,
    a.attribution_step,
    a.n_candidate_judges
FROM deposit.opinion_author_attribution a
WHERE a.gender_resolved AND a.person_unique
'''

with psycopg2.connect(**DSN) as conn:
    conn.set_session(readonly=True)
    population = pd.read_sql(POPULATION_SQL, conn)

print(f'eligible attributions: {len(population):,}')
population.groupby('attribution_step').size().rename('opinions').to_frame()

## 2. What "random" means here, and why not `random.sample`

The draw has to satisfy two things at once:

1. **Unrelated to anything about the row.** The selection must not depend on
   court, year, tier, gender or on how well the match went, or the precision
   estimate is biased.
2. **Reproducible by anyone, forever.** A reviewer must be able to re-run this
   and obtain the same 200 rows, years from now, on another machine.

A seeded shuffle satisfies (1) but not reliably (2): it depends on the library,
its version, and on row order. We tried the SQL equivalent first --
`setseed()` with `ORDER BY random()` -- and it failed outright: two runs with
the same seed returned two different samples, because PostgreSQL distributes
the scan across parallel workers and the row order feeding `random()` changes.

So the draw uses a **keyed hash of the opinion id**:

    rank within step := order by md5(opinion_id || SALT)

MD5 output is uniform and effectively independent of the input, so ordering by
it is a random permutation with respect to every attribute of the row. It is
also a pure function of the id: no row order, no RNG state, no parallelism can
change it. The same ids and the same salt give the same sample on any machine.

The pipeline-side SQL uses the same keyed-hash ordering. This notebook is the
generator of record for the released 20/60/60/60 validation sample because it
draws directly from the restored publication database.

In [ ]:
def draw_rank(opinion_id: int, salt: str = SALT) -> str:
    """Position of a row in the random permutation. Pure function of the id."""
    return hashlib.md5(f'{opinion_id}{salt}'.encode()).hexdigest()

population['draw_key'] = population.opinion_id.map(draw_rank)
population['rank_in_step'] = (
    population.groupby('attribution_step')['draw_key']
              .rank(method='first').astype(int)
)
population[['opinion_id', 'attribution_step', 'draw_key', 'rank_in_step']].head()

## 3. How many rows per step, and why not the same number

| Step | How the author is identified | Sampled | Why |
|---|---|---:|---|
| 1 | CourtListener's structured `author_id` | 20 | **Control only.** No name matching happens here -- the link is given in the source data. Annotating it measures CourtListener's data quality, not this pipeline's. Twenty rows are enough to notice if the upstream link, or the annotation procedure itself, is unsound. |
| 2 | recorded author name vs. the judge pool | 60 | Name matching, and by far the largest contributor. |
| 3 | name from the XML `<author>` element | 60 | Name matching on a cleaner signal. |
| 4 | attribution formula parsed out of HTML | 60 | Name matching on the messiest signal; its lower tiers accept surname-only matches. This is where errors are most likely. |

The allocation is neither equal nor proportional to stratum size. It is
proportional to **where the risk is**, because the quantity being estimated is
precision *per step*, not a single corpus-wide number. A corpus-weighted
figure can still be computed afterwards -- the stratum sizes travel with the
file, and `validation/score_validation_sample.py` uses them.

In [ ]:
frames = []
for step, n in ALLOCATION.items():
    stratum = population[population.attribution_step == step]
    frames.append(stratum[stratum.rank_in_step <= n])
sample = pd.concat(frames).sort_values(['attribution_step', 'opinion_id'])

stratum_sizes = population.groupby('attribution_step').size()
sample['stratum_size'] = sample.attribution_step.map(stratum_sizes)

print(f'drawn: {len(sample)} rows')
pd.DataFrame({'population': stratum_sizes,
              'sampled': sample.groupby('attribution_step').size()})

## 4. Evidence that the draw is actually unbiased

Asserting that MD5 is uniform is not evidence. The check is whether the sample
resembles the population on attributes that were **not** used to stratify it.
Filing year, court and recorded gender played no part in the selection, so if
the draw is unbiased their distributions should match within sampling error.

In [ ]:
rows = []
for step in sorted(ALLOCATION):
    pop = population[population.attribution_step == step]
    smp = sample[sample.attribution_step == step]
    rows.append({
        'step': step,
        'n_sample': len(smp),
        'year_mean_pop': pop.year_filed.mean(),
        'year_mean_sample': smp.year_filed.mean(),
        'year_median_pop': pop.year_filed.median(),
        'year_median_sample': smp.year_filed.median(),
        'share_female_pop': (pop.gender == 'f').mean(),
        'share_female_sample': (smp.gender == 'f').mean(),
        'n_courts_pop': pop.court_id.nunique(),
        'n_courts_sample': smp.court_id.nunique(),
    })
comparison = pd.DataFrame(rows).set_index('step').round(3)
comparison

In [ ]:
# A formal check on filing year: does the sample look drawn from the same
# distribution as its stratum? A small p-value would be a warning sign.
try:
    from scipy import stats
    for step in sorted(ALLOCATION):
        pop = population.loc[population.attribution_step == step, 'year_filed'].dropna()
        smp = sample.loc[sample.attribution_step == step, 'year_filed'].dropna()
        ks = stats.ks_2samp(smp, pop)
        flag = 'ok' if ks.pvalue > 0.05 else 'CHECK'
        print(f'step {step}: KS D={ks.statistic:.3f}  p={ks.pvalue:.3f}  {flag}')
except ImportError:
    print('scipy not available -- rely on the comparison table above')

## 5. Reproducibility

The draw is re-computed from scratch and compared against the sample above.
This is the claim a reviewer cares about, so it is asserted rather than
described.

In [ ]:
def redraw(pop: pd.DataFrame, allocation: dict, salt: str) -> set:
    p = pop[['opinion_id', 'attribution_step']].copy()
    p['k'] = p.opinion_id.map(lambda i: draw_rank(i, salt))
    p['r'] = p.groupby('attribution_step')['k'].rank(method='first').astype(int)
    return {tuple(x) for x in
            p[p.r <= p.attribution_step.map(allocation)][['opinion_id']].values}

first  = {(i,) for i in sample.opinion_id}
second = redraw(population, ALLOCATION, SALT)
assert first == second, 'the draw is not reproducible'
print(f'reproducible: {len(first)} identical opinion ids on re-draw')

# A different salt must give a genuinely different sample, otherwise the
# selection is not actually keyed on it.
other = redraw(population, ALLOCATION, 'some-other-salt')
print(f'overlap with a different salt: {len(first & other)} of {len(first)}')

## 6. Attach what the annotator needs to judge

For each drawn opinion: the string the match was based on, the judge it was
matched to, and the opening of the opinion, where the attribution formula
appears. HTML is preferred over plain text because that is what steps 3 and 4
read; the tags are stripped.

The excerpt is 3000 characters, not fewer. At 1200 the attribution line was
missing for 8 of 200 rows -- a long case caption and an en banc panel list
can fill that much before the opinion names its author. Those rows would
have been annotated `unclear` for want of evidence rather than because the
match was doubtful, which would have biased precision downwards.

In [ ]:
DETAIL_SQL = '''
SELECT
    a.opinion_id, a.cluster_id, ao.court_short_name,
    d.recorded_author_string    AS author_str,
    d.courtlistener_author_id   AS author_id,
    d.xml_stage_extracted_name  AS xml_extracted_author_name,
    d.html_stage_extracted_name AS html_extracted_author_name,
    BTRIM(REGEXP_REPLACE(COALESCE(d.xml_stage_author_tag, ''), '\\s+', ' ', 'g')) AS xml_author_tag,
    d.html_stage_pattern        AS html_extraction_pattern,
    -- Steps 2-4 match against a candidate pool and record the names.
    -- Step 1 has no pool: it follows CourtListener's author_id straight to
    -- the person, so the name must come from people_db_person, or the
    -- control rows reach the annotator with nothing to check against.
    -- The candidates of the channel that actually resolved the row, not the
    -- first non-null one. A row resolved in the XML stage can still carry
    -- candidates from an earlier name-stage attempt that failed, and showing
    -- those to the annotator means showing judges the match never chose.
    CASE d.assignment_channel
        WHEN 'previous_final'  THEN d.name_stage_candidates
        WHEN 'xml_author_tag'  THEN d.xml_stage_candidates
        WHEN 'html_extraction' THEN d.html_stage_candidates
    END AS matched_judge_from_pool,
    -- Step 1 has no pool: it follows CourtListener's author_id straight to
    -- the person, so the name must come from people_db_person.
    BTRIM(CONCAT_WS(' ', p.name_first, p.name_last)) AS matched_judge_from_id,
    LEFT(REGEXP_REPLACE(REGEXP_REPLACE(
        COALESCE(NULLIF(BTRIM(REGEXP_REPLACE(COALESCE(o.html_with_citations, ''),
                                             '<[^>]+>', ' ', 'g')), ''),
                 o.plain_text, ''),
        '&[a-zA-Z]+;|&#[0-9]+;', ' ', 'g'), '\\s+', ' ', 'g'), 3000) AS text_excerpt
FROM deposit.opinion_author_attribution a
JOIN deposit.opinion_author_attribution_diagnostics d USING (opinion_id)
JOIN deposit.appellate_opinion ao USING (opinion_id)
JOIN public.search_opinion o ON o.id = a.opinion_id
LEFT JOIN public.people_db_person p ON p.id = d.courtlistener_author_id
WHERE a.opinion_id = ANY(%(ids)s)
'''

ids = [int(i) for i in sample.opinion_id]
with psycopg2.connect(**DSN) as conn:
    conn.set_session(readonly=True)
    detail = pd.read_sql(DETAIL_SQL, conn, params={'ids': ids})

out = sample.merge(detail, on='opinion_id', how='left')

out['matched_judge'] = out['matched_judge_from_pool'].fillna(
    out['matched_judge_from_id'])
out = out.drop(columns=['matched_judge_from_pool', 'matched_judge_from_id'])

# What the pipeline actually matched on. This is NOT the same thing for
# every step, and conflating them was the first version's mistake:
#   step 1  matched on CourtListener's structured author_id. No name was
#           read, no extraction happened.
#   step 2  matched on the recorded author_str.
#   step 3  matched on the name pulled out of the XML <author> element.
#   step 4  matched on the name pulled out of an HTML attribution formula.
out['match_basis'] = np.select(
    [out.attribution_step == 1,
     out.attribution_step == 2,
     out.attribution_step == 3],
    ['structured author_id=' + out.author_id.astype('Int64').astype(str),
     out.author_str,
     out.xml_extracted_author_name],
    default=out.html_extracted_author_name)

# The recorded author_str is carried for every row, including step 1 where
# it played no part in the match. That is deliberate: where it is present
# and disagrees with the matched judge, it is evidence that the structured
# link may be wrong. Two such rows in this sample turned out to be genuine
# errors in CourtListener's author_id -- see
# deposit_pipeline/05_deposit/10_author_id_conflict_check.sql.
out['author_str_recorded'] = out.author_str

# Court texts carry artefacts of the documents they came from: HTML &nbsp;
# arrives as U+00A0, which looks like a space but is not one, and texts
# recovered from scans leave private-use code points behind where a font
# mapped a glyph that Unicode does not define. Both are invisible and make
# editors warn about the file. The excerpt is a reading aid, so it is
# normalised here. The deposited opinion text in public.search_opinion is
# left exactly as CourtListener published it.
INVISIBLE = re.compile(
    r'[\u0000-\u0008\u000b-\u001f\u007f-\u009f'   # control characters
    r'\u200b-\u200f\ufeff'                          # zero-width, BOM, bidi
    r'\ue000-\uf8ff]'                                # private use area
)
out['text_excerpt'] = (out.text_excerpt.astype(str)
                          .str.replace('\u00a0', ' ', regex=False)
                          .str.replace(INVISIBLE, '', regex=True)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip())

print(f'{len(out)} rows, {(out.text_excerpt == "").sum()} without an excerpt')
out[['opinion_id', 'attribution_step', 'match_basis',
     'author_str_recorded', 'matched_judge']].head()

## 7. Write the file, keeping any annotation already done

Because the draw is a pure function of the id, re-running this notebook with a
larger allocation **adds** rows and never moves the existing ones. Any verdict
already entered is carried over by opinion id, so re-drawing costs no work.

In [ ]:
ANNOTATION_COLS = ['verdict', 'correct_judge_if_incorrect',
                   'error_source', 'annotator_note']

for col in ANNOTATION_COLS:
    out[col] = pd.NA

if OUT.exists():
    previous = pd.read_csv(OUT)
    have = [c for c in ANNOTATION_COLS if c in previous.columns]
    if have:
        merged = out.drop(columns=have).merge(
            previous[['opinion_id'] + have], on='opinion_id', how='left')
        carried = merged['verdict'].notna().sum() if 'verdict' in merged else 0
        out = merged
        print(f'carried over {carried} existing verdicts')

COLUMN_ORDER = ['opinion_id', 'cluster_id', 'court_id', 'court_short_name',
                'year_filed', 'attribution_step', 'stratum_size', 'tier',
                'tier_label', 'match_basis', 'author_str_recorded', 'xml_author_tag',
                'html_extraction_pattern', 'matched_judge', 'gender',
                'text_excerpt'] + ANNOTATION_COLS
out = out[[c for c in COLUMN_ORDER if c in out.columns]]

OUT.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT, index=False)
print(f'written: {OUT}')
print(out.groupby('attribution_step').size().rename('rows').to_string())

## 8. How to annotate

Open the CSV and fill **`verdict`** for every row with one of:

| value | meaning |
|---|---|
| `correct` | the excerpt shows this judge wrote the opinion |
| `incorrect` | the excerpt names a different judge -- put that name in `correct_judge_if_incorrect` |
| `unclear` | the excerpt does not say who wrote it |

`unclear` is a real answer, not a cop-out: those rows are excluded from the
precision denominator and reported separately, which is more honest than
guessing.

For every `incorrect` row, also fill **`error_source`** with one of:

| value | meaning |
|---|---|
| `courtlistener` | the source record was wrong: `author_id` points to the wrong judge, or `author_str` names the wrong one |
| `extraction` | the wrong name was taken out of the text, for example a name from the `Before` panel line |
| `matching` | the right name was extracted but matched to the wrong judge, for example the wrong judge of two with the same surname |

Step 1 errors are `courtlistener` by definition, because that step copies
`author_id` and matches nothing. From step 2 onward the distinction separates
faults in the source from faults in this procedure, which is the more
informative half of the result.

Then score it:

```bash
python3 validation/score_validation_sample.py
```

That reports precision per step with Wilson confidence intervals, plus a
corpus-weighted figure, and writes
`validation/author_attribution_validation_precision.csv`.